# S3 STARE-PODS Demo — AWS S3 + RDS Postgres

Cloud counterpart of `local_starepods_examples.ipynb`. Runs the same six-step flow against real **AWS S3** (Parquet partitions) and a real **RDS Postgres** `PodsMetadata` table.

**Workflow**
1. Ingest a GMI granule → Parquet partitions on S3 + RDS metadata (with optional `clean_before_run`)
2. Find intersecting data for a bounding box via STARE SIDs + RDS
3. Download intersecting Parquet partitions from S3
4. Reconstitute an HDF5 file (both S1 and S2 scans)
5. Compare the reconstituted structure with the original granule
6. Verify RDS metadata

**Requires** `starepandas/.config` (next to the starepandas package) with AWS + RDS credentials. The sample granule defaults to the in-repo `tests/data/granules/1C.GPM.GMI...V07B.HDF5`; override with the `STAREPODS_SAMPLE_GRANULE` env var.

In [1]:
#import subprocess, sys
#subprocess.check_call([sys.executable, "-m", "pip", "install", "-e",
#                       "/Users/thatdaihaiton/Workspace/STARE/STAREPandas/stare_demo_and_construct_parallel",
#                       "-q"])

In [2]:
import os
import time
import h5py
from starepandas.demo_lib import StarePodsDemo
from starepandas.staredataframe import _ensure_rds_db_and_table

## Configuration

Edit these paths and parameters before running.

In [3]:
import starepandas

# AWS + RDS credentials. Resolved relative to the installed package so it works
# from any cwd (the .config lives next to the starepandas package).
CONFIG_PATH = os.path.join(os.path.dirname(os.path.abspath(starepandas.__file__)), ".config")

# Resolve the sample granule from the in-repo test-data dir so the notebook is
# safe to run anywhere (no dependency on an external sample directory). Override
# with the STAREPODS_SAMPLE_GRANULE env var to point at your own granule.
_REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(starepandas.__file__)))
GRANULE_FILE = os.environ.get(
    "STAREPODS_SAMPLE_GRANULE",
    os.path.join(
        _REPO_ROOT, "tests", "data", "granules",
        "1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5",
    ),
)

# S3 root where Parquet partitions and RDS metadata for this demo live.
S3_PREFIX = "s3://zarrpods/gmi-demo-parquet"

# STARE partition level used for both ingestion and bbox → SIDs lookup.
# Capped at MAX_PARTITION_LEVEL = 4 (~256 cells/granule), the regime
# where each Parquet partition is multi-MB — ideal for S3.
STARE_LEVEL = 10

# Bounding box filter — set to None to reconstitute the full granule
# (matching local_starepods_examples.ipynb), or e.g. (115, -30, 120, -25)
# to restrict to SW Australia / Perth.
BBOX = None   # full granule, no spatial filter — mirrors the local demo

DATASETS = ["GMI_S1", "GMI_S2"]

OUTPUT_HDF5 = "/tmp/gmi_s3_reconstituted.h5"

# Set to True to wipe S3_PREFIX (S3 objects + RDS metadata rows) before
# ingesting. Mirrors the local demo's CLEAN_BEFORE_RUN flag — prevents
# duplicate RDS rows on re-runs. Keep True unless you intentionally
# want to append more granules under the same prefix.
CLEAN_BEFORE_RUN = True

print(f"Granule  : {os.path.basename(GRANULE_FILE)}")
print(f"Datasets : {DATASETS}")
print(f"BBox     : {BBOX}  (None = full granule)")
print(f"S3 root  : {S3_PREFIX}")
print(f"Clean    : {CLEAN_BEFORE_RUN}")

Granule  : 1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5
Datasets : ['GMI_S1', 'GMI_S2']
BBox     : None  (None = full granule)
S3 root  : s3://zarrpods/gmi-demo-parquet
Clean    : True


## Step 1 — Ingest granule → S3 Parquet + RDS

In [4]:
%%time
demo = StarePodsDemo(aws_config_path=CONFIG_PATH)

s3_paths = demo.ingest_granules(
    data_path=GRANULE_FILE,
    instrument="GMI",
    s3_prefix=S3_PREFIX,
    level=STARE_LEVEL,
    clean_before_run=CLEAN_BEFORE_RUN,
)
print(f"Stored {len(s3_paths)} dataset path(s):")
for p in s3_paths:
    print(f"  {p}")

# Granule basename — used as a substring filter on group_path. Note: as of
# the quaternary pod-code layout (2026-06-14) the S3 layout is FLAT and the
# granule basename is embedded in the chunk *filename*, bracketed by '-':
#   <S3_PREFIX>/<podcode>-<granule_basename>-<dataset>.parquet
# So the old startswith(S3_PREFIX + '/' + basename) scoping no longer matches.
# We use a substring match on the basename instead.
granule_basename = os.path.splitext(os.path.basename(GRANULE_FILE))[0]
granule_path_marker = f"-{granule_basename}-"   # matches the filename-embedded span

INFO:starepandas.ingest:clean_before_run=True → wiping s3://zarrpods/gmi-demo-parquet on S3 + RDS first


INFO:starepandas.ingest:clean_s3_prefix(s3://zarrpods/gmi-demo-parquet): deleted 514 RDS row(s), 514 S3 object(s)


INFO:starepandas.ingest:Ingesting GMI granules from /Users/thatdaihaiton/Workspace/STARE/STAREPandas/stare_pods_aws_parallel/tests/data/granules/1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5


INFO:starepandas.ingest:Found 1 GMI file(s)


INFO:starepandas.ingest:Processing 1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5


...

  Progress: 200/251 partitions written...


  Progress: 250/251 partitions written...


INFO:starepandas.ingest:✓ Stored 1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5 → ['s3://zarrpods/gmi-demo-parquet', 's3://zarrpods/gmi-demo-parquet']


INFO:starepandas.ingest:Ingested 2 Parquet dataset(s)


✓ Inserted 251 metadata rows into RDS
✓ Finished writing 251 Parquet partitions to s3://zarrpods/gmi-demo-parquet
Stored 2 dataset path(s):
  s3://zarrpods/gmi-demo-parquet
  s3://zarrpods/gmi-demo-parquet
CPU times: user 11.6 s, sys: 1.06 s, total: 12.6 s
Wall time: 1min 38s


## Step 2 — Find intersecting data via STARE SIDs

In [5]:
if BBOX is not None:
    location_sids = demo.get_sids_for_bbox(*BBOX, level=STARE_LEVEL)
    print(f"Generated {len(location_sids)} SIDs for bbox {BBOX}")
    intersecting = demo.find_intersecting_data(location_sids, instruments=["GMI"])
    # Scope to our granule. Substring match on the basename, which the flat
    # pod-code layout embeds in the chunk filename (bracketed by '-').
    if not intersecting.empty and "group_path" in intersecting.columns:
        intersecting = intersecting[
            intersecting["group_path"].str.contains(granule_path_marker, regex=False)
        ]
    print(f"Found {len(intersecting)} intersecting metadata row(s).")
else:
    location_sids = None
    intersecting = None
    print("BBOX is None — Step 4 will reconstitute the full granule directly.")

if intersecting is not None and not intersecting.empty:
    intersecting[["Dataset", "grouped_id", "group_path"]].head(8)


BBOX is None — Step 4 will reconstitute the full granule directly.


## Step 3 — Download intersecting Parquet partitions from S3

In [6]:
%%time
if intersecting is not None and not intersecting.empty:
    data_dict = demo.download_and_analyze(
        intersecting,
        instruments=list(intersecting["Dataset"].unique()),
    )
    for ds_name, sdf in data_dict.items():
        print(f"{ds_name}: {len(sdf)} rows, columns: {list(sdf.columns[:6])} …")
        display(sdf.head(3))
else:
    print("No intersecting partitions to download — Step 4 will read S3 directly.")
    data_dict = {}

No intersecting partitions to download — Step 4 will read S3 directly.
CPU times: user 28 μs, sys: 8 μs, total: 36 μs
Wall time: 33.1 μs


## Step 4 — Reconstitute HDF5 (S1 + S2)

In [7]:
%%time
# s3_prefix scope: with CLEAN_BEFORE_RUN=True the bucket only holds this
# granule's data, so passing the broad S3_PREFIX is correct and avoids the
# layout mismatch the old per-granule S3 prefix would create.
recon_path = demo.reconstitute_hdf5(
    dataset=DATASETS,
    output_hdf5_path=OUTPUT_HDF5,
    bbox=BBOX,
    s3_prefix=S3_PREFIX,
)
print(f"Written to: {recon_path}")


INFO:starepandas.demo_lib:Reconstituting HDF5 for dataset='GMI_S1' over full granule (no spatial filter)


INFO:starepandas.demo_lib:Reconstituting HDF5 for dataset='GMI_S2' over full granule (no spatial filter)


INFO:starepandas.demo_lib:✓ Reconstituted HDF5 written to /tmp/gmi_s3_reconstituted.h5


Written to: /tmp/gmi_s3_reconstituted.h5
CPU times: user 10.4 s, sys: 2.58 s, total: 12.9 s
Wall time: 2min 39s


## Step 5 — Structure comparison: reconstituted vs original

In [8]:
def dump_structure(path, label):
    """Print HDF5 group/dataset tree with shapes and dtypes."""
    print(f"\n--- {label} ---")
    with h5py.File(path, "r") as f:
        def _visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"  /{name:50s} {str(obj.shape):20s} {obj.dtype}")
            elif isinstance(obj, h5py.Group) and name != "/":
                print(f"  /{name:50s} Group")
        f.visititems(_visit)

dump_structure(recon_path, f"RECONSTITUTED  ({os.path.basename(recon_path)})")
dump_structure(GRANULE_FILE, f"ORIGINAL       ({os.path.basename(GRANULE_FILE)})")


--- RECONSTITUTED  (gmi_s3_reconstituted.h5) ---
  /S1                                                 Group
  /S1/Latitude                                        (2983, 221)          float32
  /S1/Longitude                                       (2983, 221)          float32
  /S1/Quality                                         (2983, 221)          int8
  /S1/SCstatus                                        Group
  /S1/SCstatus/FractionalGranuleNumber                (2983,)              float64
  /S1/SCstatus/SCaltitude                             (2983,)              float32
  /S1/SCstatus/SClatitude                             (2983,)              float32
  /S1/SCstatus/SClongitude                            (2983,)              float32
  /S1/SCstatus/SCorientation                          (2983,)              int16
  /S1/ScanTime                                        Group
  /S1/ScanTime/DayOfMonth                             (2983,)              int8
  /S1/ScanTime/DayOfYear       

## Step 6 — RDS metadata verification

In [9]:
conn = _ensure_rds_db_and_table("StarePodsMetadata")
try:
    with conn.cursor() as cur:
        # Flat pod-code layout: the basename is embedded in the chunk
        # filename, so use a LIKE substring match (not a startswith prefix).
        cur.execute(
            'SELECT "Dataset", COUNT(*) '
            'FROM "PodsMetadata" '
            'WHERE "MetadataJson"->>%s LIKE %s '
            'GROUP BY "Dataset" ORDER BY "Dataset"',
            ("group_path", f"%{granule_path_marker}%"),
        )
        rows = cur.fetchall()
    print(f"RDS scope: group_path contains '{granule_path_marker}'")
    for ds, cnt in rows:
        print(f"  {ds}: {cnt} partition(s)")
finally:
    conn.close()


RDS scope: group_path contains '-1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B-'
  GMI_S1: 263 partition(s)
  GMI_S2: 251 partition(s)
